In [ ]:
!pip install catboost
!pip install lime

In [ ]:
# =============================================================================
# TASK 5: SHIPMENT MODE PREDICTION & OPTIMIZATION
# Dataset : DataCo Smart Supply Chain Dataset
# Target  : Shipping Mode (First Class / Second Class / Standard Class / Same Day)
# Models  : Logistic Regression, Decision Tree, Random Forest,
#           XGBoost, LightGBM, CatBoost, SVM
# Metrics : Accuracy, Macro Precision, Macro Recall, Macro F1,
#           Weighted F1, ROC-AUC (per-class and aggregate)
# XAI     : SHAP + LIME
# Author  : Rakesh S Gautham (Student ID: 1196243)
# Thesis  : LJMU MSc AI/ML, March 2026
# 1. LOAD DATASET
# 2. EDA SUMMARY
# 3. DROP PII AND IRRELEVANT COLUMNS
# 4. TARGET VARIABLE CONSTRUCTION
# 5. CATEGORICAL GROUPING & COLLINEARITY REMOVAL
# 6. OUTLIER TREATMENT (IQR Capping)
# 7. FEATURE ENGINEERING
# 8. DROPPING LEAKAGE COLUMNS
# 9. FEATURE / TARGET SEPARATION
# 10. PREPROCESSING PIPELINE
# 11. TRAIN / TEST SPLIT
# 12. MODEL DEFINITIONS
# 13. TRAINING & EVALUATION
# 14. COMPARISON TABLE
# 15. VISUALISATIONS
# 16. CLASSIFICATION REPORT — BEST MODEL
# 17. SHAP EXPLAINABILITY
# 18. LIME EXPLAINABILITY
# 19. FINAL SUMMARY
# =============================================================================

# ── 0. IMPORTS ────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from time import time
from pathlib import Path

# Preprocessing & Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, RandomizedSearchCV
)
from sklearn.preprocessing import (
    LabelEncoder, RobustScaler, OneHotEncoder, label_binarize
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)

# Statistical validation
from scipy.stats import ttest_rel

# SHAP
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("[INFO] SHAP not installed. Install with: pip install shap")

# LIME
import lime
import lime.lime_tabular

# Google Colab drive
from google.colab import drive

OUTDIR = Path("task5_outputs")
OUTDIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_JOBS       = -1
CV_FOLDS     = 5
np.random.seed(RANDOM_STATE)

print("=" * 75)
print("  TASK 5 — SHIPMENT MODE PREDICTION & OPTIMIZATION")
print("  Classes : First Class | Same Day | Second Class | Standard Class")
print("  Primary Metric : Macro F1-Score (equal weight across all four classes)")
print("=" * 75)

# ── 1. LOAD DATASET ───────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=True)
scdata_df_raw = pd.read_csv(
    '/content/drive/MyDrive/DataCoSupplyChainDataset.csv',
    encoding="latin1"
)
scdata_df = scdata_df_raw.copy()
print(f"\n[OK] Dataset loaded: {scdata_df.shape[0]:,} rows x {scdata_df.shape[1]} columns")

# ── 2. EDA SUMMARY ────────────────────────────────────────────────────────────
print("\n── 2. EDA Summary ──────────────────────────────────────────────────────")
print(f"   Shape            : {scdata_df.shape}")
print(f"   Missing values   : {scdata_df.isnull().sum().sum():,}")
print(f"   Duplicate rows   : {scdata_df.duplicated().sum():,}")
print(f"\n   Shipping Mode distribution:\n{scdata_df['Shipping Mode'].value_counts()}")

# ── 3. DROP PII AND IRRELEVANT COLUMNS ───────────────────────────────────────
print("\n── 3. Dropping PII and Irrelevant Columns ───────────────────────────────")

drop_col_list = [
    'Category Id', 'Customer Email', 'Customer Fname', 'Customer Id',
    'Customer Lname', 'Customer Password', 'Department Id',
    'Order Customer Id', 'Order Id', 'Order Item Cardprod Id',
    'Order Item Id', 'Product Card Id', 'Product Category Id',
    'Product Description', 'Product Image', 'Order Zipcode',
    'Customer Zipcode', 'Product Status', 'Customer Street'
]
scdata_df = scdata_df.drop(columns=drop_col_list, errors='ignore')

# Drop cancelled orders — ambiguous shipment mode, no meaningful label
scdata_df = scdata_df[scdata_df["Delivery Status"] != "Shipping canceled"].copy()
print(f"   Shape after PII drop and cancelled order removal: {scdata_df.shape}")

# ── 4. TARGET VARIABLE CONSTRUCTION ──────────────────────────────────────────
# =============================================================================
#   The target is the raw Shipping Mode field:
#     First Class, Same Day, Second Class, Standard Class
#   LabelEncoder maps these to integers 0–3 alphabetically.
# =============================================================================
print("\n── 4. Target Variable Construction ─────────────────────────────────────")

le = LabelEncoder()
scdata_df["shipment_mode"] = le.fit_transform(scdata_df["Shipping Mode"])

CLASS_NAMES = list(le.classes_)   # ['First Class','Same Day','Second Class','Standard Class']
N_CLASSES   = len(CLASS_NAMES)

print(f"   Classes : {CLASS_NAMES}")
dist = scdata_df["shipment_mode"].value_counts().sort_index()
for i, c in enumerate(CLASS_NAMES):
    n = dist.get(i, 0)
    print(f"   Class {i} — {c:<20}: {n:>7,}  ({n/len(scdata_df)*100:.1f}%)")

# Target variable distribution plot
fig, ax = plt.subplots(figsize=(8, 4))
cls_vals   = [dist.get(i, 0) for i in range(N_CLASSES)]
cls_colors = ["#1976D2", "#43A047", "#FFA726", "#EF5350"]
bars = ax.bar(CLASS_NAMES, cls_vals, color=cls_colors, edgecolor="white", width=0.5)
for bar, val in zip(bars, cls_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 200,
        f"{val:,}\n({val/len(scdata_df)*100:.1f}%)",
        ha="center", va="bottom", fontsize=9
    )
ax.set_ylabel("Order Count")
ax.set_title("Task 5 — Target Variable: Shipment Mode Class Distribution",
             fontsize=11, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("task5_outputs/task5_target_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n   [OK] Saved task5_outputs/task5_target_distribution.png")

# ── 5. CATEGORICAL GROUPING & COLLINEARITY REMOVAL ───────────────────────────
print("\n── 5. Categorical Grouping & Collinearity Removal ──────────────────────")

def top_n_categories(df, col, n=15):
    top_categories = df[col].value_counts().nlargest(n).index
    return df[col].apply(lambda x: x if x in top_categories else "Others")

grouping_config = {
    "Order City"    : 15,
    "Category Name" : 15,
    "Order Region"  : 15,
    "Order Country" : 10,
    "Product Name"  : 10,
    "Customer City" : 10,
    "Customer State": 15,
    "Order State"   : 15,
}
for col, n in grouping_config.items():
    if col in scdata_df.columns:
        scdata_df[f"{col} Grouped"] = top_n_categories(scdata_df, col, n)

drop_original_cols = list(grouping_config.keys())
scdata_df = scdata_df.drop(
    columns=[c for c in drop_original_cols if c in scdata_df.columns]
)
print(f"   Shape after categorical grouping: {scdata_df.shape}")

# Remove highly collinear features
scdata_df = scdata_df.drop(
    columns=["Sales per customer", "Sales", "Product Price",
             "Order Item Profit Ratio", "Benefit per order"],
    errors="ignore"
)
print(f"   Shape after collinearity removal: {scdata_df.shape}")

# ── 6. OUTLIER TREATMENT (IQR Capping) ───────────────────────────────────────
print("\n── 6. Outlier Treatment (IQR Capping) ──────────────────────────────────")

cap_cols = [
    "Order Item Discount", "Order Item Product Price",
    "Order Item Total",    "Order Profit Per Order"
]
cap_cols = [c for c in cap_cols if c in scdata_df.columns]

capping_log = []
for col in cap_cols:
    Q1, Q3 = scdata_df[col].quantile([0.25, 0.75])
    IQR     = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_lo = (scdata_df[col] < lower).sum()
    n_hi = (scdata_df[col] > upper).sum()
    scdata_df[col] = scdata_df[col].clip(lower=lower, upper=upper)
    capping_log.append({
        "Feature": col, "Lower Cap": round(lower, 2),
        "Upper Cap": round(upper, 2),
        "Capped Lower": n_lo, "Capped Upper": n_hi,
        "Total Capped": n_lo + n_hi
    })
    print(f"   {col:<40} lower={lower:.2f}  upper={upper:.2f}  "
          f"capped={n_lo + n_hi:,}")

pd.DataFrame(capping_log).to_csv(
    "task5_outputs/task5_outlier_capping_log.csv", index=False
)
print("[OK] Capping log saved → task5_outputs/task5_outlier_capping_log.csv")

# ── 7. FEATURE ENGINEERING ───────────────────────────────────────────────────
print("\n── 7. Feature Engineering ──────────────────────────────────────────────")

date_cols = ["order date (DateOrders)", "shipping date (DateOrders)"]
for col in date_cols:
    if col in scdata_df.columns:
        scdata_df[col] = pd.to_datetime(scdata_df[col], errors="coerce")

# Temporal features
if "order date (DateOrders)" in scdata_df.columns:
    scdata_df["order_month"]     = scdata_df["order date (DateOrders)"].dt.month
    scdata_df["order_dayofweek"] = scdata_df["order date (DateOrders)"].dt.dayofweek

# Order processing time (used in feature engineering only; excluded later as leakage)
if all(c in scdata_df.columns for c in date_cols):
    scdata_df["order_processing_time"] = (
        scdata_df["shipping date (DateOrders)"]
        - scdata_df["order date (DateOrders)"]
    ).dt.days.clip(lower=0)

print("   [OK] Temporal features and order processing time engineered")

# ── 8. DROPPING LEAKAGE COLUMNS ─────────────────────────────────────────────────────────
# =============================================================================
#   Shipping Mode is the target — kept only as encoded label.
#   Delivery Status and Late_delivery_risk are post-dispatch outcomes.
#   Days for shipping (real) and Days for shipment (scheduled) are post-
#   dispatch observations not available at order placement time.
#   order_processing_time is derived from shipping dates — same leakage risk.
#   All date columns dropped after feature extraction.
# =============================================================================
print("\n── 8. Leakage Guard — Dropping Post-Dispatch Columns ───────────────────")

LEAKAGE_COLS = [
    "Shipping Mode",                          # direct source of target
    "Delivery Status", "Late_delivery_risk",  # post-shipment leakage
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "order_processing_time",                  # computed from shipping dates
]
LEAKAGE_COLS = [c for c in LEAKAGE_COLS if c in scdata_df.columns]
scdata_df    = scdata_df.drop(columns=LEAKAGE_COLS)
print(f"   [OK] Leakage columns removed. Feature count: {scdata_df.shape[1] - 1}")

# ── 9. FEATURE / TARGET SEPARATION ───────────────────────────────────────────
print("\n── 9. Feature / Target Separation ──────────────────────────────────────")

y = scdata_df["shipment_mode"]
X = scdata_df.drop(columns=["shipment_mode"])

num_feats    = X.select_dtypes(include=["number"]).columns.tolist()
cat_feats    = X.select_dtypes(include=["object", "category"]).columns.tolist()
all_features = num_feats + cat_feats

print(f"   Numeric features    : {len(num_feats)}")
print(f"   Categorical features: {len(cat_feats)}")
print(f"   Total features      : {len(all_features)}")

# ── 10. PREPROCESSING PIPELINE ───────────────────────────────────────────────
print("\n── 10. Preprocessing Pipeline ──────────────────────────────────────────")

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,     num_feats),
    ("cat", categorical_transformer, cat_feats),
], remainder="drop")

# ── 11. TRAIN / TEST SPLIT ────────────────────────────────────────────────────
print("\n── 11. Train-Test Split (Stratified 80/20) ─────────────────────────────")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"   Train : {X_train.shape[0]:,} samples")
print(f"   Test  : {X_test.shape[0]:,} samples")

print("\n   Train class distribution (post-split):")
for i, c in enumerate(CLASS_NAMES):
    n = (y_train == i).sum()
    print(f"     Class {i} — {c:<20} : {n:>7,}  ({n/len(y_train)*100:.1f}%)")

# Class weights
classes           = np.unique(y_train)
cw                = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, cw))
print(f"\n   Class weights : {class_weight_dict}")

# Sample weights for XGBoost (does not accept class_weight kwarg)
sample_weights = np.array([cw[int(label)] for label in y_train])

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# SVM training subset
SVM_SAMPLE_SIZE = min(30000, len(X_train))

rng_svm = np.random.default_rng(RANDOM_STATE)

svm_sample_idx = rng_svm.choice(
    len(X_train),
    size=SVM_SAMPLE_SIZE,
    replace=False
)

X_train_svm = X_train.iloc[svm_sample_idx]
y_train_svm = y_train.iloc[svm_sample_idx]

sample_weights_svm = sample_weights[svm_sample_idx]

print(
    f"   SVM training sample : {SVM_SAMPLE_SIZE:,} "
    f"(subset of {len(X_train):,})"
)

# ── 12. MODEL DEFINITIONS ─────────────────────────────────────────────────────
print("\n── 12. Model Definitions ────────────────────────────────────────────────")

models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("clf",  LogisticRegression(
            class_weight="balanced", max_iter=1000,
            solver="lbfgs", multi_class="auto",
            random_state=RANDOM_STATE, n_jobs=N_JOBS,
        ))
    ]),
    "Decision Tree": Pipeline([
        ("prep", preprocessor),
        ("clf",  DecisionTreeClassifier(
            class_weight="balanced", max_depth=10,
            min_samples_leaf=50, random_state=RANDOM_STATE,
        ))
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("clf",  RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            max_depth=15, min_samples_leaf=20,
            n_jobs=N_JOBS, random_state=RANDOM_STATE,
        ))
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("clf",  XGBClassifier(
            n_estimators=300, learning_rate=0.05,
            max_depth=6, eval_metric="mlogloss",
            objective="multi:softprob",
            num_class=N_CLASSES,
            n_jobs=N_JOBS, random_state=RANDOM_STATE, verbosity=0,
        ))
    ]),
    "LightGBM": Pipeline([
        ("prep", preprocessor),
        ("clf",  LGBMClassifier(
            n_estimators=300, learning_rate=0.05,
            max_depth=8, num_leaves=63,
            class_weight="balanced",
            objective="multiclass", num_class=N_CLASSES,
            n_jobs=N_JOBS, random_state=RANDOM_STATE, verbose=-1,
        ))
    ]),
    "CatBoost": Pipeline([
        ("prep", preprocessor),
        ("clf",  CatBoostClassifier(
            iterations=300, learning_rate=0.05,
            depth=6, auto_class_weights="Balanced",
            random_seed=RANDOM_STATE, verbose=0,
        ))
    ]),
    "SVM": Pipeline([
        ("prep", preprocessor),
        ("clf", CalibratedClassifierCV(
            estimator=LinearSVC(
                class_weight="balanced",
                dual=True,
                random_state=RANDOM_STATE
            ),
            method="isotonic",
            cv=5
        ))
    ]),

}
print(f"   {len(models)} models configured for four-class classification.")

# ── 13. TRAINING & EVALUATION ─────────────────────────────────────────────────
print("\n── 13. Training & Evaluation ────────────────────────────────────────────")

results = {}

for model_name, pipeline in models.items():
    print(f"\n   Training → {model_name} ...", end=" ", flush=True)
    t0 = time()

    # 13a. Cross-validation — Macro F1 as primary CV metric
    # Macro F1 gives equal weight to all four shipment mode classes
    if model_name == "SVM":
        cv_scores = cross_val_score(
            pipeline, X_train_svm, y_train_svm,
            cv=skf, scoring="f1_macro", n_jobs=N_JOBS
        )
    else:
        cv_scores = cross_val_score(
            pipeline, X_train, y_train,
            cv=skf, scoring="f1_macro", n_jobs=N_JOBS
        )

    # 13b. Fit on full training set
    # XGBoost does not accept class_weight so sample_weight is used at fit time
    if model_name == "XGBoost":
        pipeline.fit(X_train, y_train, clf__sample_weight=sample_weights)
    elif model_name == "SVM":
    # SVM fitted on stratified sample for computational feasibility
    # Test evaluation uses full X_test for fair comparison
        pipeline.fit(X_train_svm, y_train_svm)
    else:
        pipeline.fit(X_train, y_train)
    train_time = time() - t0

    # 13c. Predictions
    y_pred = pipeline.predict(X_test)
    clf    = pipeline.named_steps["clf"]

    # All models support predict_proba (SVM via CalibratedClassifierCV)
    if hasattr(clf, "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)
    else:
        raise ValueError(
            f"{model_name}: predict_proba not available. "
            f"Wrap with CalibratedClassifierCV."
        )

    # 13d. Metrics — full suite consistent
    acc         = accuracy_score(y_test, y_pred)
    macro_prec  = precision_score(y_test, y_pred, average="macro",    zero_division=0)
    macro_rec   = recall_score(y_test, y_pred,    average="macro",    zero_division=0)
    macro_f1    = f1_score(y_test, y_pred,        average="macro",    zero_division=0)
    weighted_f1 = f1_score(y_test, y_pred,        average="weighted", zero_division=0)
    per_cls_f1  = f1_score(y_test, y_pred,        average=None,       zero_division=0)

    # Multiclass ROC-AUC using One-vs-Rest macro averaging
    roc_auc = roc_auc_score(
        y_test, y_prob,
        multi_class="ovr",
        average="macro",
        labels=classes
    )

    results[model_name] = {
        "Accuracy"          : round(acc,              4),
        "Macro Precision"   : round(macro_prec,       4),
        "Macro Recall"      : round(macro_rec,        4),
        "Macro F1"          : round(macro_f1,         4),
        "Weighted F1"       : round(weighted_f1,      4),
        "F1 First Class"    : round(per_cls_f1[0],    4),
        "F1 Same Day"       : round(per_cls_f1[1],    4),
        "F1 Second Class"   : round(per_cls_f1[2],    4),
        "F1 Standard Class" : round(per_cls_f1[3],    4),
        "ROC-AUC"           : round(roc_auc,          4),
        "CV F1 Mean"        : round(cv_scores.mean(), 4),
        "CV F1 Std"         : round(cv_scores.std(),  4),
        "Train Time (s)"    : round(train_time,        1),
        "_pipeline"         : pipeline,
        "_y_pred"           : y_pred,
        "_y_prob"           : y_prob,
    }

    print(
        f"done [{train_time:.1f}s]  "
        f"Acc={acc:.4f}  MacroF1={macro_f1:.4f}  "
        f"ROC-AUC={roc_auc:.4f}  "
        f"CV-F1={cv_scores.mean():.4f}±{cv_scores.std():.4f}"
    )

# ── 14. COMPARISON TABLE ──────────────────────────────────────────────────────
print("\n")
print("=" * 120)
print("  MODEL COMPARISON TABLE — TASK 5: SHIPMENT MODE PREDICTION")
print("=" * 120)

metrics_cols = [
    "Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1",
    "F1 First Class", "F1 Same Day", "F1 Second Class", "F1 Standard Class",
    "ROC-AUC", "CV F1 Mean", "CV F1 Std", "Train Time (s)"
]

comparison_df = pd.DataFrame(
    {m: {k: results[m][k] for k in metrics_cols} for m in results}
).T.reset_index().rename(columns={"index": "Model"})

comparison_df = comparison_df.sort_values("Macro F1", ascending=False).reset_index(drop=True)
comparison_df.insert(0, "Rank", comparison_df.index + 1)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4f}".format)

print(comparison_df.to_string(index=False))
print("=" * 120)

best_model_name = comparison_df.iloc[0]["Model"]
best_macro_f1   = comparison_df.iloc[0]["Macro F1"]
best_acc        = comparison_df.iloc[0]["Accuracy"]
best_roc        = comparison_df.iloc[0]["ROC-AUC"]

print(f"\n   Best Model  : {best_model_name}")
print(f"   Macro F1    : {best_macro_f1:.4f}")
print(f"   Accuracy    : {best_acc:.4f}")
print(f"   ROC-AUC     : {best_roc:.4f}")
print("=" * 120)

comparison_df.to_csv("task5_outputs/task5_model_comparison.csv", index=False)
print("\n[OK] Comparison table saved → task5_outputs/task5_model_comparison.csv")

# Convenience references for the best pipeline
best_pipeline = results[best_model_name]["_pipeline"]
best_y_pred   = results[best_model_name]["_y_pred"]
best_y_prob   = results[best_model_name]["_y_prob"]

# ── 15. VISUALISATIONS ────────────────────────────────────────────────────────
print("\n── 15. Generating Visualisations ───────────────────────────────────────")

PALETTE     = sns.color_palette("Set2", n_colors=len(models))
MODEL_ORDER = comparison_df["Model"].tolist()

# ── 15a. Grouped bar chart — core metrics ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
bar_metrics = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1"]
x     = np.arange(len(MODEL_ORDER))
width = 0.15

for i, metric in enumerate(bar_metrics):
    vals = [results[m][metric] for m in MODEL_ORDER]
    bars = ax.bar(x + i * width, vals, width, label=metric, color=PALETTE[i])
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=6.5, rotation=90
        )

ax.set_xticks(x + width * 2)
ax.set_xticklabels(MODEL_ORDER, rotation=25, ha="right", fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Task 5 — Model Comparison: Core Metrics",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("task5_outputs/task5_model_comparison_bars.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved task5_outputs/task5_model_comparison_bars.png")

# ── 15b. Per-class F1 grouped bar chart ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
per_class_metrics = [
    "F1 First Class", "F1 Same Day",
    "F1 Second Class", "F1 Standard Class"
]
per_class_colors = cls_colors  # reuse the four class colours defined above
x2     = np.arange(len(MODEL_ORDER))
width2 = 0.20

for i, (metric, color) in enumerate(zip(per_class_metrics, per_class_colors)):
    vals = [results[m][metric] for m in MODEL_ORDER]
    bars = ax.bar(x2 + i * width2, vals, width2,
                  label=metric.replace("F1 ", ""), color=color, alpha=0.85)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=6.5, rotation=90
        )

ax.set_xticks(x2 + width2 * 1.5)
ax.set_xticklabels(MODEL_ORDER, rotation=25, ha="right", fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("F1-Score")
ax.set_title(
    "Task 5 — Per-Class F1 Score: "
    "First Class | Same Day | Second Class | Standard Class",
    fontsize=12, fontweight="bold"
)
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("task5_outputs/task5_per_class_f1.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved task5_outputs/task5_per_class_f1.png")

# ── 15c. Heatmap — all metrics ────────────────────────────────────────────────
heatmap_cols = [
    "Accuracy", "Macro F1", "Weighted F1",
    "F1 First Class", "F1 Same Day",
    "F1 Second Class", "F1 Standard Class",
    "ROC-AUC", "CV F1 Mean"
]
hm_df = comparison_df.set_index("Model")[heatmap_cols].astype(float)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(hm_df, annot=True, fmt=".4f", cmap="YlGn",
            linewidths=0.5, ax=ax, annot_kws={"size": 8.0})
ax.set_title("Task 5 — Model Performance Heatmap",
             fontsize=13, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("task5_outputs/task5_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved task5_outputs/task5_heatmap.png")

# ── 15d. Confusion matrix — best model ────────────────────────────────────────
cm   = confusion_matrix(y_test, best_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_model_name}",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("task5_outputs/task5_confusion_matrix_best.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved task5_outputs/task5_confusion_matrix_best.png")

# ── 15e. Confusion matrices for all models  ─────────────────────────────────
n_models = len(MODEL_ORDER)
ncols    = 4
nrows    = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4.5))
axes = axes.flatten()

for i, model_name in enumerate(MODEL_ORDER):
    cm_i   = confusion_matrix(y_test, results[model_name]["_y_pred"])
    disp_i = ConfusionMatrixDisplay(
        confusion_matrix=cm_i, display_labels=CLASS_NAMES
    )
    disp_i.plot(ax=axes[i], colorbar=False, cmap="Blues")
    axes[i].set_title(
        f"{model_name}\nMacro F1={results[model_name]['Macro F1']:.4f}",
        fontsize=9, fontweight="bold"
    )
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")
    axes[i].tick_params(axis='x', rotation=30)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Task 5 — Confusion Matrices: All Models",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("task5_outputs/task5_confusion_matrices_all.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved task5_outputs/task5_confusion_matrices_all.png")

# ── 15f. ROC curves — one subplot per shipment mode class (OvR) ───────────────
y_test_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))

fig, axes = plt.subplots(1, N_CLASSES, figsize=(5 * N_CLASSES, 5), sharey=True)
for class_idx, class_name in enumerate(CLASS_NAMES):
    ax = axes[class_idx]
    for i, model_name in enumerate(MODEL_ORDER):
        y_prob_model = results[model_name]["_y_prob"]
        fpr, tpr, _  = roc_curve(y_test_bin[:, class_idx], y_prob_model[:, class_idx])
        roc_auc_c    = auc(fpr, tpr)
        ax.plot(fpr, tpr,
                label=f"{model_name} ({roc_auc_c:.3f})",
                color=PALETTE[i], lw=1.6)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_title(f"OvR — {class_name}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=6.5, loc="lower right")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("True Positive Rate")
fig.suptitle("Task 5 — ROC Curves (One-vs-Rest) by Shipment Mode",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("task5_outputs/task5_roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved task5_outputs/task5_roc_curves.png")

# ── 15g. CV Macro F1 horizontal bar chart ─────────────────────────────────────
cv_means = [results[m]["CV F1 Mean"] for m in MODEL_ORDER]
cv_stds  = [results[m]["CV F1 Std"]  for m in MODEL_ORDER]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(MODEL_ORDER, cv_means, xerr=cv_stds,
        color=PALETTE[:len(MODEL_ORDER)], height=0.55,
        capsize=5, ecolor="gray")
ax.set_xlabel("CV Macro F1-Score (mean ± std)")
ax.set_title("Task 5 — Cross-Validated Macro F1 (5-Fold)",
             fontsize=12, fontweight="bold")
ax.set_xlim(0, 1.05)
for i, (m, s) in enumerate(zip(cv_means, cv_stds)):
    ax.text(m + s + 0.01, i, f"{m:.4f}", va="center", fontsize=8)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("task5_outputs/task5_cv_macro_f1.png", dpi=150, bbox_inches="tight")
plt.close()
print("   [✓] Saved task5_outputs/task5_cv_macro_f1.png")

# ── 16. CLASSIFICATION REPORT — BEST MODEL ────────────────────────────────────
print(f"\n── 16. Classification Report — {best_model_name} ────────────────────────")
print(classification_report(
    y_test, best_y_pred,
    target_names=CLASS_NAMES,
    digits=4
))

# #############################################################################
# TASK 5 — TEST DATA VALIDATION
# Shipment Mode Prediction (First Class / Same Day / Second Class / Standard Class)
# #############################################################################

print("\n")
print("=" * 90)
print("  TEST DATA VALIDATION — TASK 5: SHIPMENT MODE PREDICTION")
print(f"  Best Model : {best_model_name}")
print("=" * 90)

y_val_pred_t5 = results[best_model_name]["_y_pred"]
y_val_prob_t5 = results[best_model_name]["_y_prob"]
y_test_arr_t5 = np.array(y_test)

# ── V1. Re-confirm test set metrics ───────────────────────────────────────────
print("\n   ── V1. Test Set Metric Confirmation ────────────────────────────────")

val_acc_t5 = accuracy_score(y_test_arr_t5, y_val_pred_t5)
val_mf1_t5 = f1_score(y_test_arr_t5, y_val_pred_t5,
                       average="macro", zero_division=0)
val_wf1_t5 = f1_score(y_test_arr_t5, y_val_pred_t5,
                       average="weighted", zero_division=0)
val_auc_t5 = roc_auc_score(
    y_test_arr_t5, y_val_prob_t5,
    multi_class="ovr", average="macro", labels=classes
)
per_cls_f1_t5  = f1_score(y_test_arr_t5, y_val_pred_t5,
                            average=None, zero_division=0)
per_cls_rec_t5 = recall_score(y_test_arr_t5, y_val_pred_t5,
                               average=None, zero_division=0)
per_cls_pre_t5 = precision_score(y_test_arr_t5, y_val_pred_t5,
                                  average=None, zero_division=0)

print(f"   Test Accuracy    : {val_acc_t5:.4f}")
print(f"   Test Macro F1    : {val_mf1_t5:.4f}")
print(f"   Test Weighted F1 : {val_wf1_t5:.4f}")
print(f"   Test ROC-AUC     : {val_auc_t5:.4f}")
print(f"\n   Per-class breakdown:")
print(f"   {'Shipment Mode':<20} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("   " + "-" * 55)
for i, name in enumerate(CLASS_NAMES):
    print(f"   {name:<20} {per_cls_pre_t5[i]:>10.4f} "
          f"{per_cls_rec_t5[i]:>10.4f} {per_cls_f1_t5[i]:>10.4f}")

# ── V2. CV vs Test gap — overfitting check ────────────────────────────────────
print("\n   ── V2. Overfitting Check: CV vs Test Score Gap ─────────────────────")

cv_mean_t5 = results[best_model_name]["CV F1 Mean"]
cv_std_t5  = results[best_model_name]["CV F1 Std"]
gap_t5     = abs(cv_mean_t5 - val_mf1_t5)

print(f"   CV Macro F1 Mean : {cv_mean_t5:.4f} ± {cv_std_t5:.4f}")
print(f"   Test Macro F1    : {val_mf1_t5:.4f}")
print(f"   Gap              : {gap_t5:.4f}")

if gap_t5 < 0.02:
    verdict_t5 = "Excellent — model generalises well, no overfitting detected"
elif gap_t5 < 0.05:
    verdict_t5 = "Acceptable — minor gap, model is stable"
else:
    verdict_t5 = "Warning — notable gap between CV and test, review regularisation"

print(f"   Verdict          : {verdict_t5}")
# ── V3. Paired t-test — best model vs all others ──────────────────────────────
# Uses a 30k stratified sample for speed — Sample size is sufficient for t-test validity.
from sklearn.model_selection import StratifiedShuffleSplit
print("\n   ── V3. Statistical Validation (Paired t-test) ──────────────────────")
print(f"   Comparing all models against best model: {best_model_name}")
print(f"   Note: CV uses 30k stratified sample for computational efficiency.")

TTEST_SAMPLE = 30_000
sss_tt = StratifiedShuffleSplit(
    n_splits=1, test_size=None,
    train_size=TTEST_SAMPLE,
    random_state=RANDOM_STATE
)
tt_idx, _ = next(sss_tt.split(X_train, y_train))
X_tt = X_train.iloc[tt_idx]
y_tt = y_train.iloc[tt_idx]

# 5-fold CV on the sample — consistent fold count with main evaluation
skf_tt = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

best_cv_t5 = cross_val_score(
    results[best_model_name]["_pipeline"],
    X_tt, y_tt,
    cv=skf_tt, scoring="f1_macro", n_jobs=N_JOBS
)

print(f"\n   {'Model':<22}  {'CV Macro F1':>12}  {'p-value':>10}  {'Significant':>13}")
print("   " + "-" * 64)

for model_name in MODEL_ORDER:
    if model_name == best_model_name:
        continue
    other_cv = cross_val_score(
        results[model_name]["_pipeline"],
        X_tt, y_tt,
        cv=skf_tt, scoring="f1_macro", n_jobs=N_JOBS
    )
    _, p = ttest_rel(best_cv_t5, other_cv)
    sig  = "Yes (p<0.05)" if p < 0.05 else "No"
    mean = other_cv.mean()
    print(f"   {model_name:<22}  {mean:>12.4f}  {p:>10.4f}  {sig:>13}")

# ── V4. Same Day class — detailed validation ──────────────────────────────────
# Same Day is the highest-priority SLA class and typically the smallest;
print("\n   ── V4. Same Day Class Validation (Highest SLA Priority) ────────────")

same_day_id    = CLASS_NAMES.index("Same Day")
sd_actual      = (y_test_arr_t5 == same_day_id).sum()
sd_predicted   = (y_val_pred_t5 == same_day_id).sum()
sd_tp          = ((y_val_pred_t5 == same_day_id) & (y_test_arr_t5 == same_day_id)).sum()
sd_fn          = ((y_val_pred_t5 != same_day_id) & (y_test_arr_t5 == same_day_id)).sum()
sd_fp          = ((y_val_pred_t5 == same_day_id) & (y_test_arr_t5 != same_day_id)).sum()

print(f"   Actual Same Day orders        : {sd_actual:>7,}")
print(f"   Predicted Same Day orders     : {sd_predicted:>7,}")
print(f"   Correctly caught (TP)         : {sd_tp:>7,}")
print(f"   Missed Same Day (FN)          : {sd_fn:>7,}  "
      f"← orders that missed SLA without alert")
print(f"   False alarms (FP)             : {sd_fp:>7,}")
print(f"   Same Day Recall               : {per_cls_rec_t5[same_day_id]:.4f}  "
      f"({'Good' if per_cls_rec_t5[same_day_id] > 0.5 else 'Needs improvement — consider threshold tuning'})")

# ── V5. Misclassification pattern — which modes get confused ──────────────────
print("\n   ── V5. Misclassification Pattern (Confusion Analysis) ───────────────")

cm_t5     = confusion_matrix(y_test_arr_t5, y_val_pred_t5)
cm_norm   = cm_t5.astype(float) / cm_t5.sum(axis=1, keepdims=True)

print(f"\n   Normalised confusion matrix (row = actual, col = predicted):")
print(f"\n   {'':>20}", end="")
for name in CLASS_NAMES:
    print(f"  {name[:10]:>12}", end="")
print()
print("   " + "-" * 75)
for i, actual_name in enumerate(CLASS_NAMES):
    print(f"   {actual_name:<20}", end="")
    for j in range(N_CLASSES):
        marker = " ←" if i == j else "  "
        print(f"  {cm_norm[i, j]:>11.3f}{marker[0]}", end="")
    print()

print(f"\n   Most confused shipment mode pairs:")
confused_pairs = []
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if i != j and cm_t5[i, j] > 0:
            confused_pairs.append((cm_t5[i, j], CLASS_NAMES[i], CLASS_NAMES[j]))
confused_pairs.sort(reverse=True)
for count, actual, predicted in confused_pairs[:5]:
    print(f"   {actual:<20} → predicted as {predicted:<20} : {count:,} orders")

# ── V6. Prediction confidence per class ──────────────────────────────────────
print("\n   ── V6. Prediction Confidence per Class ──────────────────────────────")

for class_id, class_name in enumerate(CLASS_NAMES):
    pred_as_this = (y_val_pred_t5 == class_id)
    if pred_as_this.sum() > 0:
        mean_conf = y_val_prob_t5[pred_as_this, class_id].mean()
        high_conf = (y_val_prob_t5[pred_as_this, class_id] >= 0.80).sum()
        pct_high  = 100 * high_conf / pred_as_this.sum()
        print(f"   {class_name:<20} mean confidence={mean_conf:.4f}  "
              f"high conf (≥0.80)={pct_high:.1f}%")

# ── V7. Confidence distribution plot (one panel per class) ───────────────────
fig, axes = plt.subplots(1, N_CLASSES, figsize=(5 * N_CLASSES, 4))
for class_id, (class_name, color) in enumerate(zip(CLASS_NAMES, cls_colors)):
    ax = axes[class_id]
    ax.hist(
        y_val_prob_t5[:, class_id],
        bins=30, color=color,
        edgecolor="white", alpha=0.85
    )
    ax.axvline(0.5, color="black", linestyle="--",
               lw=1.2, label="0.5 threshold")
    ax.set_xlabel(f"P({class_name})")
    ax.set_ylabel("Frequency")
    ax.set_title(
        f"{class_name}\nF1={per_cls_f1_t5[class_id]:.4f}  "
        f"Recall={per_cls_rec_t5[class_id]:.4f}",
        fontsize=9, fontweight="bold"
    )
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

fig.suptitle(
    f"Task 5 — Prediction Confidence by Shipment Mode | {best_model_name}",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(
    "task5_outputs/task5_validation_confidence.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("\n   [✓] Saved task5_outputs/task5_validation_confidence.png")

# ── V8. Normalised confusion matrix heatmap ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_norm,
    annot=True, fmt=".3f",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 10},
)
ax.set_xlabel("Predicted Shipment Mode", fontsize=10)
ax.set_ylabel("Actual Shipment Mode", fontsize=10)
ax.set_title(
    f"Task 5 — Normalised Confusion Matrix\n{best_model_name}",
    fontsize=11, fontweight="bold"
)
plt.xticks(rotation=30, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(
    "task5_outputs/task5_validation_confusion_normalised.png",
    dpi=150, bbox_inches="tight"
)
plt.close()
print("   [✓] Saved task5_outputs/task5_validation_confusion_normalised.png")

# ── V9. Validation summary ────────────────────────────────────────────────────
print(f"""
   ── V9. Validation Summary ───────────────────────────────────────────────
   Best Model         : {best_model_name}
   Test Accuracy      : {val_acc_t5:.4f}
   Test Macro F1      : {val_mf1_t5:.4f}
   Test Weighted F1   : {val_wf1_t5:.4f}
   Test ROC-AUC       : {val_auc_t5:.4f}
   CV vs Test Gap     : {gap_t5:.4f}  ({verdict_t5.split(' — ')[0]})

   Per-class F1:""")
for i, name in enumerate(CLASS_NAMES):
    print(f"     {name:<20} : {per_cls_f1_t5[i]:.4f}")

print(f"""
   Same Day Recall    : {per_cls_rec_t5[same_day_id]:.4f}
   Same Day Missed    : {sd_fn:,} orders not assigned to priority channel


   ─────────────────────────────────────────────────────────────────────────
""")
print("=" * 90)

# ── 17. SHAP EXPLAINABILITY ────────────────────────────────────────────────────
# Per-class summary dot plots, per-class bar plots, and global bar

if SHAP_AVAILABLE:
    print("\n── 17. SHAP Explainability ──────────────────────────────────────────────")
    try:
        best_clf = best_pipeline.named_steps["clf"]
        prep     = best_pipeline.named_steps["prep"]

        N_SHAP        = min(500, X_test.shape[0])
        rng           = np.random.default_rng(RANDOM_STATE)
        idx           = rng.choice(X_test.shape[0], N_SHAP, replace=False)
        X_shap_sample = X_test.iloc[idx]

        X_shap_tr  = prep.transform(X_shap_sample)
        ohe_cols   = (prep.named_transformers_["cat"]["ohe"]
                      .get_feature_names_out(cat_feats).tolist())
        feat_names = num_feats + ohe_cols
        shap_df    = pd.DataFrame(X_shap_tr, columns=feat_names)

        if hasattr(best_clf, "feature_importances_"):
            explainer   = shap.TreeExplainer(best_clf)
            shap_values = explainer.shap_values(X_shap_tr)
        else:
            explainer   = shap.Explainer(best_clf, X_shap_tr)
            shap_values = explainer(X_shap_tr).values

        # Normalise to list of arrays — one per class
        if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
            sv_list = [shap_values[:, :, c] for c in range(N_CLASSES)]
        elif isinstance(shap_values, list):
            sv_list = shap_values
        else:
            sv_list = [shap_values]

        # Per-class summary dot plots and bar plots
        for cls_idx, cls_label in enumerate(CLASS_NAMES):
            if cls_idx >= len(sv_list):
                break
            sv = sv_list[cls_idx]

            # Summary dot plot
            plt.figure(figsize=(10, 7))
            shap.summary_plot(sv, shap_df, max_display=15,
                              show=False, plot_type="dot")
            plt.title(
                f"SHAP Summary — {best_model_name} | "
                f"Class: {cls_label}",
                fontsize=11
            )
            plt.tight_layout()
            safe = cls_label.replace(" ", "_").lower()
            fname = f"task5_outputs/task5_shap_summary_{safe}.png"
            plt.savefig(fname, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"   [✓] Saved {fname}")

            # Bar plot per class
            plt.figure(figsize=(10, 6))
            shap.summary_plot(sv, shap_df, max_display=15,
                              show=False, plot_type="bar")
            plt.title(
                f"SHAP Feature Importance Bar — {cls_label}\n"
                f"(Model: {best_model_name})",
                fontsize=11
            )
            plt.tight_layout()
            fname_bar = f"task5_outputs/task5_shap_bar_{safe}.png"
            plt.savefig(fname_bar, dpi=150, bbox_inches="tight")
            plt.close()
            print(f"   [✓] Saved {fname_bar}")

        # Global bar — mean |SHAP| across all four classes
        if len(sv_list) >= N_CLASSES:
            mean_abs_shap    = np.mean(
                [np.abs(sv_list[i]) for i in range(N_CLASSES)], axis=0
            )
            mean_per_feature = mean_abs_shap.mean(axis=0)
            top15_idx        = np.argsort(mean_per_feature)[-15:][::-1]
            top15_feat       = [feat_names[i] for i in top15_idx]
            top15_vals       = mean_per_feature[top15_idx]

            fig, ax = plt.subplots(figsize=(10, 6))
            ax.barh(top15_feat[::-1], top15_vals[::-1],
                    color="#1976D2", alpha=0.85)
            ax.set_xlabel(
                "Mean |SHAP value| (averaged across all four shipment mode classes)"
            )
            ax.set_title(
                f"SHAP Global Feature Importance — {best_model_name} (Task 5)",
                fontsize=11, fontweight="bold"
            )
            ax.grid(axis="x", alpha=0.3)
            plt.tight_layout()
            plt.savefig("task5_outputs/task5_shap_global_bar.png",
                        dpi=150, bbox_inches="tight")
            plt.close()
            print("   [✓] Saved task5_outputs/task5_shap_global_bar.png")

    except Exception as e:
        print(f"   [!] SHAP failed for {best_model_name}: {e}")
        import traceback; traceback.print_exc()
else:
    print("\n── 17. SHAP — SKIPPED (shap not installed) ──────────────────────────────")

# ── 18. LIME EXPLAINABILITY ───────────────────────────────────────────────────
# One confidently correct instance per shipment mode class
# Probability scores in figure title
# CSV saved per class
# Missed Same Day instance — most operationally costly error

print("\n── 18. LIME Explainability ──────────────────────────────────────────────")

X_train_encoded = X_train[all_features].copy()
X_test_encoded  = X_test[all_features].copy()

cat_indices       = [all_features.index(c) for c in cat_feats if c in all_features]
categorical_names = {}

for idx_c in cat_indices:
    col_name = all_features[idx_c]
    uniq     = list(X_train_encoded[col_name].astype(str).unique())
    categorical_names[idx_c] = uniq
    X_train_encoded[col_name] = X_train_encoded[col_name].astype(str).apply(
        lambda v: uniq.index(v) if v in uniq else 0
    )
    X_test_encoded[col_name] = X_test_encoded[col_name].astype(str).apply(
        lambda v: uniq.index(v) if v in uniq else 0
    )

X_train_np = X_train_encoded.values.astype(float)
X_test_np  = X_test_encoded.values.astype(float)

def lime_predict_fn(raw_rows: np.ndarray) -> np.ndarray:
    """Wraps the full pipeline for LIME; returns class probabilities (n x 4)."""
    df_temp = pd.DataFrame(raw_rows, columns=all_features)
    for idx_c in cat_indices:
        col_name = all_features[idx_c]
        uniq     = categorical_names[idx_c]
        df_temp[col_name] = df_temp[col_name].apply(
            lambda v: uniq[int(round(v))]
            if 0 <= int(round(v)) < len(uniq) else uniq[0]
        )
    return best_pipeline.predict_proba(df_temp)

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data         = X_train_np,
    feature_names         = all_features,
    class_names           = CLASS_NAMES,
    categorical_features  = cat_indices,
    categorical_names     = categorical_names,
    mode                  = "classification",
    random_state          = RANDOM_STATE,
    discretize_continuous = True,
)
print(f"   LIME explainer built on {X_train_np.shape[0]:,} training samples.")

y_pred_arr  = results[best_model_name]["_y_pred"]
y_prob_best = results[best_model_name]["_y_prob"]
y_test_arr  = y_test.values

num_lime_features = min(15, len(all_features))
print(f"   LIME num_features set to: {num_lime_features}")

# ── 18a. One confidently correct instance per shipment mode class ──────────────
print("\n   Generating one local explanation per shipment mode class (correct predictions)...")

for class_id, class_name in enumerate(CLASS_NAMES):
    correct_mask = (y_test_arr == class_id) & (y_pred_arr == class_id)
    correct_idxs = np.where(correct_mask)[0]

    if len(correct_idxs) == 0:
        any_idxs = np.where(y_test_arr == class_id)[0]
        if len(any_idxs) == 0:
            print(f"   [!] No instances found for {class_name} — skipping")
            continue
        chosen = any_idxs[0]
    else:
        # Most confident correct prediction
        chosen = correct_idxs[np.argmax(y_prob_best[correct_idxs, class_id])]

    instance_raw = X_test_np[chosen]
    prob_scores  = y_prob_best[chosen]

    exp = lime_explainer.explain_instance(
        data_row     = instance_raw,
        predict_fn   = lime_predict_fn,
        num_features = num_lime_features,
        labels       = (class_id,),
        num_samples  = 1000,
    )

    # Print top features
    print(f"\n   Top features — {class_name}:")
    for feat, weight in exp.as_list(label=class_id):
        print(f"   {feat:<45} {weight:>+.4f}")

    # Save CSV
    safe_name = class_name.replace(" ", "_").lower()
    lime_rows = []
    for feat, weight in exp.as_list(label=class_id):
        lime_rows.append({
            "Shipment Mode"    : class_name,
            "Feature Condition": feat,
            "LIME Weight"      : round(weight, 4),
            "Direction"        : "increases probability" if weight > 0
                                 else "decreases probability",
        })
    pd.DataFrame(lime_rows).to_csv(
        f"task5_outputs/task5_lime_{safe_name}_features.csv", index=False
    )
    print(f"   [✓] Saved CSV for {class_name}")

    # Save HTML
    exp.save_to_file(f"task5_outputs/task5_lime_{safe_name}.html")

    # Save PNG — probability scores in title
    fig = exp.as_pyplot_figure(label=class_id)
    fig.set_size_inches(12, 6)
    prob_str = "  ".join(
        [f"P({CLASS_NAMES[i]})={prob_scores[i]:.3f}" for i in range(N_CLASSES)]
    )
    fig.suptitle(
        f"LIME Local Explanation — {class_name} | "
        f"Model: {best_model_name}\n{prob_str}",
        fontsize=9, fontweight="bold",
    )
    plt.tight_layout()
    png_path = f"task5_outputs/task5_lime_{safe_name}.png"
    plt.savefig(png_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   [✓] Saved {png_path}")

# ── 18b. Missed Same Day instance — most operationally costly error ────────────
# A Same Day order predicted as Standard Class means an SLA breach
# with no operational escalation
print("\n   Generating explanation for missed Same Day instance...")

# Prefer misclassified as Standard Class (worst SLA downgrade)
standard_id      = CLASS_NAMES.index("Standard Class")
missed_sd_idx    = np.where(
    (y_pred_arr == standard_id) & (y_test_arr == same_day_id)
)[0]

# Fall back to any Same Day missed
if len(missed_sd_idx) == 0:
    missed_sd_idx = np.where(
        (y_pred_arr != same_day_id) & (y_test_arr == same_day_id)
    )[0]

if len(missed_sd_idx) > 0:
    chosen_miss  = missed_sd_idx[0]
    instance_raw = X_test_np[chosen_miss]
    predicted_as = int(y_pred_arr[chosen_miss])
    prob_scores  = y_prob_best[chosen_miss]

    exp_fn = lime_explainer.explain_instance(
        data_row     = instance_raw,
        predict_fn   = lime_predict_fn,
        num_features = num_lime_features,
        num_samples  = 1000,
        labels       = (same_day_id,)  # explain why model missed Same Day
    )
    exp_fn.save_to_file("task5_outputs/task5_lime_missed_same_day.html")

    fig = exp_fn.as_pyplot_figure(label=same_day_id)
    fig.set_size_inches(12, 6)
    prob_str = "  ".join(
        [f"P({CLASS_NAMES[i]})={prob_scores[i]:.3f}" for i in range(N_CLASSES)]
    )
    fig.suptitle(
        f"LIME — Missed Same Day Order "
        f"(predicted as {CLASS_NAMES[predicted_as]})\n"
        f"Model: {best_model_name} | {prob_str}",
        fontsize=10,
        fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig("task5_outputs/task5_lime_missed_same_day.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print("   [✓] Saved task5_outputs/task5_lime_missed_same_day.png")
else:
    print("   [!] No missed Same Day instances found in test set.")

print("\n   ── LIME Summary ──────────────────────────────────────────────────────")
print("   Local : One representative instance per shipment mode class")
print("   Error : One missed Same Day instance (most operationally costly error)")
print("   LIME operates in raw feature space via pipeline wrapper.")

# ── 19. FINAL SUMMARY ────────────────────────────────────────────────────────
print("\n")
print("=" * 120)
print("  FINAL SUMMARY — TASK 5: SHIPMENT MODE PREDICTION")
print("=" * 120)
print(comparison_df[[
    "Rank", "Model", "Accuracy",
    "Macro Precision", "Macro Recall",
    "Macro F1", "Weighted F1",
    "F1 First Class", "F1 Same Day",
    "F1 Second Class", "F1 Standard Class",
    "ROC-AUC", "CV F1 Mean", "CV F1 Std", "Train Time (s)"
]].to_string(index=False))
print("=" * 120)
print(f"\n   Best Performing Model : {best_model_name}")
print(f"   Macro F1              : {best_macro_f1:.4f}")
print(f"   Accuracy              : {best_acc:.4f}")
print(f"   ROC-AUC               : {best_roc:.4f}")
print(f"\n   Saved outputs:")
print("    • task5_outputs/task5_target_distribution.png")
print("    • task5_outputs/task5_outlier_capping_log.csv")
print("    • task5_outputs/task5_model_comparison.csv")
print("    • task5_outputs/task5_model_comparison_bars.png")
print("    • task5_outputs/task5_per_class_f1.png")
print("    • task5_outputs/task5_heatmap.png")
print("    • task5_outputs/task5_confusion_matrix_best.png")
print("    • task5_outputs/task5_confusion_matrices_all.png")
print("    • task5_outputs/task5_roc_curves.png")
print("    • task5_outputs/task5_cv_macro_f1.png")
if SHAP_AVAILABLE:
    print("    • task5_outputs/task5_shap_summary_[first_class/same_day/...].png")
    print("    • task5_outputs/task5_shap_bar_[first_class/same_day/...].png")
    print("    • task5_outputs/task5_shap_global_bar.png")
print("    • task5_outputs/task5_lime_[mode].png/.html")
print("    • task5_outputs/task5_lime_[mode]_features.csv")
print("    • task5_outputs/task5_lime_missed_same_day.png/.html")
print("=" * 120)


In [ ]:
from google.colab import files
!zip -r task5_outputs.zip /content/task5_outputs
files.download("task5_outputs.zip")